# Full-text Model with Contrastive Learning

This notebook test full-text model using contrastive learning where training and inference are in the same corpus.

# Set up

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Class definition

In [ ]:
import tensorflow as tf

class FeedForward(tf.keras.layers.Layer):
  def __init__(self, d_model, dropout_rate=0.1):
    super().__init__()
    self.seq = tf.keras.Sequential([
      tf.keras.layers.Dense(d_model, activation='relu'),
      tf.keras.layers.Dense(d_model),
      tf.keras.layers.Dropout(dropout_rate)
    ])
    self.add = tf.keras.layers.Add()
    self.layer_norm = tf.keras.layers.LayerNormalization()

  def call(self, x):
    x = self.add([x, self.seq(x)])
    x = self.layer_norm(x)
    return x


class FinetuneEncoder(tf.keras.layers.Layer):
  def __init__(self, ff_layers, dropout_rate=0.1):
    super().__init__()
    self.num_blocks = ff_layers
    self.ff_layers = [FeedForward(d_model, dropout_rate) for d_model in self.num_blocks]
    self.embedding = tf.keras.layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1))

  def call(self, x):
    for layer in self.ff_layers:
      x = layer(x)
    x = self.embedding(x)
    return x


class DistanceLayer(tf.keras.layers.Layer):
  def __init__(self, **kwargs):
      super().__init__(**kwargs)

  def call(self, x1, x2):
      distance = tf.reduce_sum(tf.square(x1 - x2), -1)
      return tf.reshape(distance, (-1,1))

from sklearn.metrics import f1_score

def f1(x,y,thres):
    yx = np.zeros(x.shape[0])
    yy = np.ones(y.shape[0])
    yhx = np.zeros(x.shape[0])
    yhx[x > thres] = 1
    yhy = np.zeros(y.shape[0])
    yhy[y > thres] = 1
    return f1_score(np.concatenate([yx,yy]), np.concatenate([yhx,yhy]))

In [ ]:
def run_model(answers, gpt1, gpt2, learning_rate=1e-5, epochs=50, batch_size=256, seeds=(1111,2222,3333,4444,5555,6666,7777,8888,9999,1010)):
  valid_accuracies = []
  valid_f1s = []
  test_accuracies = []
  test_f1s = []

  for seed in seeds:
    np.random.seed(seed)

    #train-test-validation split
    sampling_indices = np.random.uniform(0,1,answers.shape[0])
    train_answers = answers[sampling_indices < 0.8]
    train_gpt1 = gpt1[sampling_indices < 0.8]
    train_gpt2 = gpt2[sampling_indices < 0.8]
    trainX = [np.vstack([train_answers, train_gpt2]), np.vstack([train_gpt1, train_gpt1])]
    trainY = np.vstack([np.ones([train_answers.shape[0], 1]), np.zeros([train_answers.shape[0], 1])])

    valid_answers = answers[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]
    valid_gpt1 = gpt1[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]
    valid_gpt2 = gpt2[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]
    validX = [np.vstack([valid_answers, valid_gpt2]), np.vstack([valid_gpt1, valid_gpt1])]
    validY = np.vstack([np.ones([valid_answers.shape[0], 1]), np.zeros([valid_answers.shape[0], 1])])

    test_answers = answers[sampling_indices >= 0.9]
    test_gpt1 = gpt1[sampling_indices >= 0.9]
    test_gpt2 = gpt2[sampling_indices >= 0.9]


    #build model
    emb_shape = 768
    embedding = FinetuneEncoder([768]*3)
    input_1 = tf.keras.layers.Input(name="anchor", shape=[emb_shape])
    input_2 = tf.keras.layers.Input(name="positive", shape=[emb_shape])
    distances = DistanceLayer()(embedding(input_1), embedding(input_2))
    distance_network = tf.keras.Model(inputs=[input_1, input_2], outputs=distances)

    #training
    distance_network.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate),
        loss=tf.keras.losses.MeanSquaredError(),
        metrics=[tf.keras.metrics.MeanSquaredError()],
    )

    distance_network.fit(x=trainX, y=trainY, batch_size=batch_size, epochs=epochs, validation_data=[validX, validY])

    #valid accuracy
    valid_emb_answers = embedding(valid_answers)
    valid_emb_gpt1 = embedding(valid_gpt1)
    valid_emb_gpt2 = embedding(valid_gpt2)
    valid_gpt_distances = ((valid_emb_gpt1 - valid_emb_gpt2)**2).numpy().sum(axis=1)
    valid_answer_distances = ((valid_emb_gpt1 - valid_emb_answers)**2).numpy().sum(axis=1)
    valid_accuracies.append((valid_gpt_distances < valid_answer_distances).mean())

    #tune threshold
    thresholds = []
    perf = []
    for thres in np.arange(0,1,0.01):
      thresholds.append(thres)
      perf.append(f1(valid_gpt_distances, valid_answer_distances, thres))

    triplet_thres = thresholds[perf.index(max(perf))]

    #valid f1
    valid_f1s.append(max(perf))

    #test accuracy
    test_emb_answers = embedding(test_answers)
    test_emb_gpt1 = embedding(test_gpt1)
    test_emb_gpt2 = embedding(test_gpt2)
    test_gpt_distances = ((test_emb_gpt1 - test_emb_gpt2)**2).numpy().sum(axis=1)
    test_answer_distances = ((test_emb_gpt1 - test_emb_answers)**2).numpy().sum(axis=1)
    test_accuracies.append((test_gpt_distances < test_answer_distances).mean())

    #test f1
    test_f1s.append(f1(test_gpt_distances, test_answer_distances, triplet_thres))
  return valid_accuracies, test_accuracies, valid_f1s, test_f1s

## NQ Data

In [ ]:
dataset = 'SQUAD_GPT4'

import pickle

with open(f'Data/{dataset}_answer_embs.obj', 'rb') as f:
  answers = pickle.load(f)
with open(f'Data/{dataset}_gpt1_embs.obj', 'rb') as f:
  gpt1 = pickle.load(f)
with open(f'Data/{dataset}_gpt2_embs.obj', 'rb') as f:
  gpt2 = pickle.load(f)

In [ ]:
valid_accuracies, test_accuracies, valid_f1s, test_f1s = run_model(answers,gpt1,gpt2,learning_rate=1e-4, epochs=100)

In [ ]:
np.mean(valid_accuracies), np.mean(test_accuracies), np.mean(valid_f1s), np.mean(test_f1s)